In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
!pip install -qq langchain-groq langchain-community langchain-experimental duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.2/504.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 79.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-s

In [4]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00


In [5]:
from langchain.tools import tool

# LLM Setup

In [6]:
from kaggle_secrets import UserSecretsClient
from langchain_groq import ChatGroq
import os

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api_key
)

# Python Repl Tool
### run python codes

In [7]:
from langchain_experimental.utilities import PythonREPL

In [8]:
python_repl = PythonREPL()

@tool
def python_repl_tool(code : str) -> str : 
    """A Python shell.

    Use this to execute python commands.

    Input should be a valid python command.

    If you want to see the output of a value, you should print it out with `print(...)`.
    """
    return python_repl.run(code)

In [9]:
code ="""
n=10
for i n range(n):
    print(f"Current value : {i}")
"""

result = python_repl_tool.invoke({"code":code})
print(result)

Python REPL can execute arbitrary code. Use with caution.


SyntaxError('invalid syntax', ('<string>', 2, 7, 'for i n range(n):\n', 2, 8))


In [10]:
code = """
def factorial_for_loop(n):
    if n < 0:
        return "Factorial is not defined for negative numbers"
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

# Example
number = 5
print(f"Factorial of {number} is: {factorial_for_loop(number)}")   
"""

result = python_repl_tool.invoke(code)
print(result)

Factorial of 5 is: 120



# Shell Tool

In [11]:
from langchain_community.tools import ShellTool

In [12]:
shell_tool = ShellTool()

print(shell_tool.run({"commands": ["echo 'Hello World!'", "pwd", "whoami", "free -m"]}))

Executing command:
 ["echo 'Hello World!'", 'pwd', 'whoami', 'free -m']
Hello World!
/kaggle/working
root
               total        used        free      shared  buff/cache   available
Mem:           32103        1011       27579           1        3512       30634
Swap:              0           0           0



/usr/local/lib/python3.12/dist-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


# DuckDuckGo Search

## DuckDuckGoSearchRun

In [13]:
from langchain_community.tools import DuckDuckGoSearchResults, DuckDuckGoSearchRun
import pandas as pd

In [14]:
search = DuckDuckGoSearchRun()
result = search.invoke("What is the capital of India")
print(result)

India, officially the Republic ofIndia, [j][20] is a country in South Asia. It is the seventh-largest country by area; the most populous country since 2023; [21] and, since its independence in 1947, the world's most populous democracy. [22][23][24] Bounded by the Indian Ocean on the south, the Arabian Sea on the southwest, and the Bay of Bengal on the southeast, it shares land borders with ... Discover whether Delhi or New Delhi is the truecapitalofIndia. Understand their historical roots, administrative differences, and constitutional roles in this detailed and informative comparison. Learn the complete list of Indian states andcapitals, along with 8 Union Territories and theircapitals. Important for any competitive exams. Get interesting information and facts aboutIndia'scapitalcity, New Delhi -- its history, government, geography, climate and urban structure. Discover detailed information about the States andCapitalsofIndia. Learn about all 28 states, 8 Union Territories, and theirc

## DuckDuckGoSearchResults

### output_format = "list"

In [15]:
search = DuckDuckGoSearchResults(output_format="list")

answer = search.invoke("what is the formula to convert Celsius to Fahrenheit")
answer

[{'snippet': 'Home›Conversion›Temperature›CelsiustoFahrenheit(°C to °F).When boiling water on a mountain above sea level the boiling point is reduced below 100 °C. The symbol ofCelsiusdegrees is °C.Fahrenheit.',
  'title': 'CelsiustoFahrenheit(°C to °F)Conversion',
  'link': 'https://www.rapidtables.com/convert/temperature/celsius-to-fahrenheit.html'},
 {'snippet': "HowtoconvertCelsiustoFahrenheit.Formulaand example.Let's usetheformulatoconvertthe boiling point of water, i.e., 100°C intoFahrenheit. Placing the values in the firstformula, we get",
  'title': 'CelsiustoFahrenheitConverter',
  'link': 'https://www.omnicalculator.com/conversion/celsius-to-fahrenheit'},
 {'snippet': 'ConvertFahrenheittoCelsiuswith the °C = (°F - 32) x.The 1.8 in theCelsiustoFahrenheitformulais shorthand for the fraction 9/5 (since 9 divided by 5 equals 1.8).',
  'title': 'HowtoConvertCelsiustoFahrenheit:Formula&ConversionTable',
  'link': 'https://www.wikihow.com/Convert-Celsius-(°C)-to-Fahrenheit-(°F)'},
 

In [16]:
df = pd.DataFrame(answer)
df

,snippet,title,link
0,Home›Conversion›Temperature›CelsiustoFahrenhei...,CelsiustoFahrenheit(°C to °F)Conversion,https://www.rapidtables.com/convert/temperatur...
1,HowtoconvertCelsiustoFahrenheit.Formulaand exa...,CelsiustoFahrenheitConverter,https://www.omnicalculator.com/conversion/cels...
2,ConvertFahrenheittoCelsiuswith the °C = (°F - ...,HowtoConvertCelsiustoFahrenheit:Formula&Conver...,https://www.wikihow.com/Convert-Celsius-(°C)-t...
3,TemperatureConversion>Celsiusconversion(ºC) >C...,CelsiustoFahrenheitconversion: ºC to ºF calcul...,https://www.metric-conversions.org/temperature...


In [17]:
for k in df['snippet']:
    print("\n",k)


 Home›Conversion›Temperature›CelsiustoFahrenheit(°C to °F).When boiling water on a mountain above sea level the boiling point is reduced below 100 °C. The symbol ofCelsiusdegrees is °C.Fahrenheit.

 HowtoconvertCelsiustoFahrenheit.Formulaand example.Let's usetheformulatoconvertthe boiling point of water, i.e., 100°C intoFahrenheit. Placing the values in the firstformula, we get

 ConvertFahrenheittoCelsiuswith the °C = (°F - 32) x.The 1.8 in theCelsiustoFahrenheitformulais shorthand for the fraction 9/5 (since 9 divided by 5 equals 1.8).

 TemperatureConversion>Celsiusconversion(ºC) >CelsiustoFahrenheit.CelsiustoFahrenheitconversionis probably the most confusingconversionthere is, but a simple °C to °Fconversionis actually quite easy – just double the °C figure and add 30.


### backend = "news"

In [18]:
s = DuckDuckGoSearchResults(backend="news",output_format='list')
print(s)

api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text') backend='news' output_format='list'


In [19]:
result = s.invoke("Corona Virus")

df = pd.DataFrame(result)
df

,snippet,title,link,date,source
0,"When the X user shared the post in June 2013, ...",Why 2013 coronavirus warning on X likely wasn'...,https://www.msn.com/en-us/health/medical/why-2...,2026-03-17T08:07:05+00:00,Snopes on MSN
1,The Dutch health council is recommending that ...,Coronavirus vaccination age set to rise from 6...,https://www.dutchnews.nl/2026/03/coronavirus-v...,2026-03-11T08:07:05+00:00,DutchNews.nl
2,X user @Marco_Acortes and Wikimedia Commons Th...,Fact Check: Why 2013 coronavirus warning on X ...,https://www.yahoo.com/news/articles/fact-check...,2026-03-16T13:15:00+00:00,Yahoo
3,The Centers for Disease Control and Prevention...,CDC recommends coronavirus vaccine with a new ...,https://www.washingtonpost.com/health/2025/10/...,2025-10-06T00:01:00+00:00,The Washington Post


# Google Scholar

In [20]:
!pip install -q google-search-results>=2.4.2

In [21]:
from langchain_community.tools.google_scholar import GoogleScholarQueryRun
from langchain_community.utilities.google_scholar import GoogleScholarAPIWrapper

In [22]:
serp_api = user_secrets.get_secret("SERP_API_KEY")

scholar_wrapper = GoogleScholarAPIWrapper(
    serp_api_key = serp_api,
    top_k_results = 5, #default 10
    hl = "en"
)

search_tool = GoogleScholarQueryRun(api_wrapper = scholar_wrapper)
results = search_tool.invoke("LLM")
print(results)

Title: A survey on llm-as-a-judge
Authors: J Gu,X Jiang,Z Shi,H Tan,X Zhai,C Xu,W Li,Y Shen
Summary: J Gu, X Jiang, Z Shi, H Tan, X Zhai, C Xu, W Li, Y Shen… - The Innovation, 2024 - cell.com
Total-Citations: 1126

Title: From generation to judgment: Opportunities and challenges of llm-as-a-judge
Authors: D Li,B Jiang,L Huang,A Beigi,C Zhao
Summary: D Li, B Jiang, L Huang, A Beigi, C Zhao… - Proceedings of the …, 2025 - aclanthology.org
Total-Citations: 426

Title: Llm agents for education: Advances and applications
Authors: Z Chu,S Wang,J Xie,T Zhu,Y Yan,J Ye
Summary: Z Chu, S Wang, J Xie, T Zhu, Y Yan, J Ye… - arXiv preprint arXiv …, 2025 - aclanthology.org
Total-Citations: 137

Title: Llm fine-tuning: Concepts, opportunities, and challenges
Authors: XK Wu,M Chen,W Li,R Wang,L Lu,J Liu
Summary: XK Wu, M Chen, W Li, R Wang, L Lu, J Liu… - Big Data and Cognitive …, 2025 - mdpi.com
Total-Citations: 73

Title: Using an llm to help with code understanding
Authors: D Nam,A Macvean,V Hellen

# Custom Tools

# 1.Using @tool decorator

In [23]:
@tool
def squareroot(a : int) -> int :
    """Calculate the square root of the given number."""
    return a**0.5

In [24]:
root = squareroot.invoke({'a':25})  #pass input as a dictionary
print(root)

5.0


In [25]:
print(squareroot.name)
print(squareroot.description)
print(squareroot.args)

squareroot
Calculate the square root of the given number.
{'a': {'title': 'A', 'type': 'integer'}}


## what llm sees when tool is passed

In [26]:
schema = squareroot.args_schema.model_json_schema()
for k,v in schema.items():
    print(f"{k} : {v}")

description : Calculate the square root of the given number.
properties : {'a': {'title': 'A', 'type': 'integer'}}
required : ['a']
title : squareroot
type : object


# 2.Using StructuredTool

In [28]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [36]:
class DivideInput(BaseModel):
    a : int = Field(required = True, description = "dividend")
    b : int = Field(required = True, description = "divisor", gt=0)

/tmp/ipykernel_55/1533806131.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a : int = Field(required = True, description = "dividend")
/tmp/ipykernel_55/1533806131.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b : int = Field(required = True, description = "divisor", gt=0)


In [37]:
def divide_func(a : int, b : int) :
    return a/b

In [38]:
divide_tool = StructuredTool.from_function(
    func = divide_func,
    name = "divide",
    description = "Divide two numbers",
    args_schema = DivideInput
)

In [52]:
result = divide_tool.invoke({'a':10, 'b':4})
print(f"10 divided by 4 = {result}")

try :
    result = divide_tool.invoke({'a':5, 'b':0})
    print(f"5 divided by 0 = {result}")
    
except Exception as e:
    print(f"\nError : {e}")

10 divided by 4 = 2.5

Error : 1 validation error for DivideInput
b
  Input should be greater than 0 [type=greater_than, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than


In [41]:
res = divide_tool.args_schema.model_json_schema()
for k,v in res.items():
    print(f"{k} : {v}")

properties : {'a': {'description': 'dividend', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'divisor', 'exclusiveMinimum': 0, 'required': True, 'title': 'B', 'type': 'integer'}}
required : ['a', 'b']
title : DivideInput
type : object


# Toolkit

In [68]:
from langchain.tools import tool

In [69]:
@tool
def capitalize_first(a : str) -> str:
    """capitalize the first letter"""
    return a.capitalize()

@tool
def lower_case(a : str) -> str:
    """Converts a string into lower case"""
    return a.lower()

@tool
def upper_case(a : str) -> str:
    """Converts a string into upper case"""
    return a.upper()

In [70]:
class StringToolkit:
    def get_tools(self):
        return [capitalize_first, lower_case, upper_case]

In [71]:
toolkit = StringToolkit()
tools = toolkit.get_tools()

tools

[StructuredTool(name='capitalize_first', description='capitalize the first letter', args_schema=<class 'langchain_core.utils.pydantic.capitalize_first'>, func=<function capitalize_first at 0x7b5db0d76b60>),
 StructuredTool(name='lower_case', description='Converts a string into lower case', args_schema=<class 'langchain_core.utils.pydantic.lower_case'>, func=<function lower_case at 0x7b5db0d64400>),
 StructuredTool(name='upper_case', description='Converts a string into upper case', args_schema=<class 'langchain_core.utils.pydantic.upper_case'>, func=<function upper_case at 0x7b5d94ca3ba0>)]

In [72]:
for tool in tools:
    print(f"{tool.name} --> {tool.description}")

capitalize_first --> capitalize the first letter
lower_case --> Converts a string into lower case
upper_case --> Converts a string into upper case


In [76]:
test = "hELLO. this is a test string"
print(test)

output = tools[0].invoke(test)
print(f"\nCapitalize first letter : {output}")

output = tools[1].invoke(test)
print(f"Lower case : {output}")

output = tools[2].invoke(test)
print(f"Upper Case : {output}")

hELLO. this is a test string

Capitalize first letter : Hello. this is a test string
Lower case : hello. this is a test string
Upper Case : HELLO. THIS IS A TEST STRING
